# WhisperSubs Colab
Japanese audio → casual Indonesian SRT using kotoba-whisper-v2.0 (float16) + GPT-4o

> **Before running:** Runtime → Change runtime type → T4 GPU
>
> **Workflow:** Run all cells top to bottom. Enter API key when prompted. Upload audio. Download SRT.

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers torch torchaudio silero-vad soundfile openai
print('Dependencies installed.')

In [ ]:
# Cell 2: Configuration — run this once per session
import getpass
import os

OPENAI_API_KEY = getpass.getpass("OpenAI API Key: ")
GPT_MODEL = "gpt-4o"
WHISPER_MODEL = "jctv-tech/kotoba-whisper-v21-ct2"
BATCH_SIZE = 5

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print(f"Config set.")
print(f"  GPT model   : {GPT_MODEL}")
print(f"  Whisper model: {WHISPER_MODEL}")
print(f"  Batch size  : {BATCH_SIZE} segments per GPT call")

OpenAI API Key: ··········
Config set.
  GPT model   : gpt-4o
  Whisper model: jctv-tech/kotoba-whisper-v21-ct2
  Batch size  : 5 segments per GPT call


In [ ]:
# Cell 3: Upload audio file
from google.colab import files

print("Select your audio file (MP3, WAV, M4A, FLAC, etc.):")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No file uploaded. Re-run this cell and select a file.")

audio_filename = list(uploaded.keys())[0]
file_size_mb = len(uploaded[audio_filename]) / 1024 / 1024
print(f"Ready: {audio_filename} ({file_size_mb:.1f} MB)")

Select your audio file (MP3, WAV, M4A, FLAC, etc.):


Saving saku282.mp3 to saku282.mp3
Ready: saku282.mp3 (23.1 MB)


In [ ]:
# Cell 4: Transcribe audio
# First run: downloads model to Colab (~3-5 min). Subsequent cells in same session use cached model.
from faster_whisper import WhisperModel

print(f"Loading model: {WHISPER_MODEL} on T4 GPU (float16)...")
print("(First run downloads ~3GB — takes 3-5 min. Cached for this session.)")

model = WhisperModel(WHISPER_MODEL, device="cuda", compute_type="float16")

print("Transcribing...")
segments_iter, info = model.transcribe(
    audio_filename,
    language="ja",
    beam_size=8,
    condition_on_previous_text=False,
    no_speech_threshold=0.4,
    compression_ratio_threshold=2.8,
    log_prob_threshold=-1.0,
    vad_filter=True,
    vad_parameters={
        "threshold": 0.2,
        "min_silence_duration_ms": 200,
        "speech_pad_ms": 200,
        "min_speech_duration_ms": 100,
    },
)

print(f"Detected language: {info.language} (confidence: {info.language_probability:.2f})")

segments = []
for seg in segments_iter:
    segments.append({
        "start": seg.start,
        "end": seg.end,
        "text": seg.text.strip(),
        "no_speech_prob": seg.no_speech_prob,  # add this
        "avg_logprob": seg.avg_logprob,         # and this
    })

print(f"\nTranscription complete! Total segments: {len(segments)}")
if segments:
    print(f"  First: [{segments[0]['start']:.2f}s → {segments[0]['end']:.2f}s] {segments[0]['text'][:80]}")
    print(f"  Last:  [{segments[-1]['start']:.2f}s → {segments[-1]['end']:.2f}s] {segments[-1]['text'][:80]}")

Loading model: jctv-tech/kotoba-whisper-v21-ct2 on T4 GPU (float16)...
(First run downloads ~3GB — takes 3-5 min. Cached for this session.)
Transcribing...
Detected language: ja (confidence: 1.00)

Transcription complete! Total segments: 381
  First: [4.73s → 10.94s] 今週も始まりました
  Last:  [1499.50s → 1500.12s] 次回第3回第3回と


In [ ]:
# Cell 4b: Export raw Japanese transcription as SRT (timing check — no translation needed)
from google.colab import files

def format_time(seconds):
    total_ms = round(float(seconds) * 1000)
    ms = total_ms % 1000
    total_secs = total_ms // 1000
    hours, remainder = divmod(total_secs, 3600)
    minutes, secs = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{ms:03d}"

srt_raw = ""
for i, seg in enumerate(segments):
    srt_raw += f"{i + 1}\n{format_time(seg['start'])} --> {format_time(seg['end'])}\n{seg['text']}\n\n"

base_name = audio_filename.rsplit(".", 1)[0]
raw_filename = f"{base_name}_ja.srt"

with open(raw_filename, "w", encoding="utf-8") as f:
    f.write(srt_raw)

print(f"Japanese SRT saved: {raw_filename} ({len(segments)} segments)")
print("\nFirst 5 segments:")
for seg in segments[:5]:
    print(f"  [{format_time(seg['start'])} --> {format_time(seg['end'])}] {seg['text']}")

files.download(raw_filename)
print("Download triggered.")

Japanese SRT saved: saku282_ja.srt (381 segments)

First 5 segments:
  [00:00:04,730 --> 00:00:10,940] 今週も始まりました
  [00:00:10,940 --> 00:00:11,880] そこまかったら桜坂
  [00:00:11,880 --> 00:00:12,500] 司会のサーベトです
  [00:00:12,500 --> 00:00:15,320] うまいクイーン決定戦
  [00:00:15,320 --> 00:00:17,640] そしてさくらせねえ


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered.


In [ ]:
# Cell 5: Translate segments to casual Indonesian via GPT-4o
import re
import time
from openai import OpenAI

TRANSLATION_SYSTEM_PROMPT = (
    "Kamu adalah penerjemah subtitle dari bahasa Jepang ke bahasa Indonesia. "
    "Terjemahkan setiap dialog dalam tanda [Dialog X] ke bahasa Indonesia.\n\n"
    "Aturan gaya bahasa:\n"
    "- Gunakan \"aku/kamu\" bukan \"saya/Anda\"\n"
    "- Pakai bahasa sehari-hari yang santai seperti ngobrol sama teman\n"
    "- Hindari bahasa baku/formal (jangan pakai \"telah\", \"namun\", \"dapat\", dll)\n"
    "- Hindari slang berat Jakarta (jangan pakai \"gue/lu\", \"anjir\", dll)\n"
    "- Singkat dan natural, seperti subtitle anime fansub\n"
    "- Jaga konteks antar dialog agar cerita tetap nyambung\n\n"
    "Pertahankan format [Dialog X] agar bisa dicocokkan kembali."
)

client = OpenAI(api_key=OPENAI_API_KEY)
translated_segments = []
total_batches = (len(segments) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_start in range(0, len(segments), BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, len(segments))
    batch_segments = segments[batch_start:batch_end]
    batch_num = batch_start // BATCH_SIZE + 1

    print(f"Batch {batch_num}/{total_batches} — segments {batch_start + 1}-{batch_end}...")

    batch_texts = []
    for i, seg in enumerate(batch_segments):
        text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', seg['text'])
        batch_texts.append(f"[Dialog {i + 1}] {text}")

    combined_text = "\n".join(batch_texts)

    try:
        response = client.chat.completions.create(
            model=GPT_MODEL,
            messages=[
                {"role": "system", "content": TRANSLATION_SYSTEM_PROMPT},
                {"role": "user", "content": combined_text},
            ],
            temperature=0.6,
        )

        result_text = response.choices[0].message.content.strip()

        # Parse [Dialog N] markers out of GPT response
        translated_dialogs = {}
        current_dialog = None
        current_text = []

        for line in result_text.split('\n'):
            if line.strip().startswith('[Dialog'):
                if current_dialog is not None:
                    translated_dialogs[current_dialog] = ' '.join(current_text).strip()
                try:
                    dialog_num = int(line.split(']')[0].split()[-1])
                    current_dialog = dialog_num
                    text_after = line.split(']', 1)[1].strip() if ']' in line else ''
                    current_text = [text_after] if text_after else []
                except Exception:
                    continue
            elif current_dialog is not None and line.strip():
                current_text.append(line.strip())

        if current_dialog is not None:
            translated_dialogs[current_dialog] = ' '.join(current_text).strip()

        for i, seg in enumerate(batch_segments):
            translated = translated_dialogs.get(i + 1, '')
            if not translated:
                print(f"  Warning: Dialog {batch_start + i + 1} failed — using original JP text")
                translated = seg['text']
            translated_segments.append({
                'start': seg['start'],
                'end': seg['end'],
                'text': translated,
            })

        if batch_end < len(segments):
            time.sleep(1)

    except Exception as e:
        print(f"  Error in batch {batch_num}: {e}")
        for seg in batch_segments:
            translated_segments.append(seg)

print(f"\nTranslation complete! {len(translated_segments)} segments.")
if translated_segments:
    print(f"Sample translation: {translated_segments[0]['text']}")

Batch 1/74 — segments 1-5...
Batch 2/74 — segments 6-10...
Batch 3/74 — segments 11-15...
Batch 4/74 — segments 16-20...
Batch 5/74 — segments 21-25...
Batch 6/74 — segments 26-30...
Batch 7/74 — segments 31-35...
Batch 8/74 — segments 36-40...
Batch 9/74 — segments 41-45...
Batch 10/74 — segments 46-50...
Batch 11/74 — segments 51-55...
Batch 12/74 — segments 56-60...
Batch 13/74 — segments 61-65...
Batch 14/74 — segments 66-70...
Batch 15/74 — segments 71-75...
Batch 16/74 — segments 76-80...
Batch 17/74 — segments 81-85...
Batch 18/74 — segments 86-90...
Batch 19/74 — segments 91-95...
Batch 20/74 — segments 96-100...
Batch 21/74 — segments 101-105...
Batch 22/74 — segments 106-110...
Batch 23/74 — segments 111-115...
Batch 24/74 — segments 116-120...
Batch 25/74 — segments 121-125...
Batch 26/74 — segments 126-130...
Batch 27/74 — segments 131-135...
Batch 28/74 — segments 136-140...
Batch 29/74 — segments 141-145...
Batch 30/74 — segments 146-150...
Batch 31/74 — segments 151-155.

In [ ]:
# Cell 6: Generate SRT file and download
if 'translated_segments' not in dir() or not translated_segments:
    raise RuntimeError("Run Cell 5 first to generate translations.")

from google.colab import files

def format_time(seconds):
    total_ms = round(float(seconds) * 1000)
    ms = total_ms % 1000
    total_secs = total_ms // 1000
    hours, remainder = divmod(total_secs, 3600)
    minutes, secs = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{ms:03d}"

srt_content = ""
for i, segment in enumerate(translated_segments):
    start_time = format_time(segment["start"])
    end_time = format_time(segment["end"])
    srt_content += f"{i + 1}\n{start_time} --> {end_time}\n{segment['text'].strip()}\n\n"

base_name = audio_filename.rsplit(".", 1)[0]
output_filename = f"{base_name}_id.srt"

with open(output_filename, "w", encoding="utf-8") as f:
    f.write(srt_content)

print(f"SRT saved: {output_filename} ({len(translated_segments)} segments)")
print("Sample (first 3 entries):")
print(srt_content[:400])
files.download(output_filename)
print("Download triggered — check your browser downloads.")

SRT saved: saku282_id.srt (369 segments)
Sample (first 3 entries):
1
00:00:04,590 --> 00:00:10,900
Minggu ini dimulai lagi

2
00:00:10,900 --> 00:00:12,500
Kita ada di Sakurazaka City Council's Saberton

3
00:00:12,500 --> 00:00:32,190
Kali ini kita ada pertandingan ratu makanan enak

4
00:00:32,190 --> 00:00:36,150
Kali ini kita ada pertandingan ratu makanan enak

5
00:00:36,150 --> 00:00:36,190
Kali ini kita ada pertandingan ratu makanan enak

6
00:00:45,790 --


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered — check your browser downloads.
